# 105. Construct Binary Tree from Preorder and Inorder Traversal
**Difficulty:** 🟡 Medium · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/

## 💡 Concepts

**Core concept(s):** Use two traversal orders together, with a **hash map** for fast lookups.

**Why it applies here:** In **preorder** the first value is always the root. In **inorder** everything left of the root is its left subtree, everything right is its right subtree. So the first preorder value splits inorder into two halves — recurse on each. A hash map makes the split instant.

**Key intuition:** Preorder gives you the root; inorder tells you which values fall left vs right of it.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

### 📚 What is a Hash Map (used here)?
A **hash map** (`dict`) gives **O(1)** lookups. We map each value to its position in inorder, so finding where the root splits the array is instant instead of an O(n) scan.

---

**Prerequisite knowledge:**
- Recursion.
- A `dict` from value -> index.

## 📝 Problem

Rebuild the unique binary tree from its `preorder` and `inorder` value lists (all values distinct).

**Example**
```
preorder = [3,9,20,15,7]
inorder  = [9,3,15,20,7]   -> tree [3,9,20,None,None,15,7]
```

> Two approaches: naive index-search `O(n²)` and hash-map `O(n)`.

In [1]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Search + Slice (worst)

**Idea:** Root = `preorder[0]`. Find it in `inorder` (a scan), split into left/right halves, and recurse on slices.

**Time:** `O(n²)` — an `O(n)` search + slicing at each of n nodes.

**Space:** `O(n²)` from slicing (plus recursion).

In [ ]:
def build_naive(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    if not preorder:
        return None
    root = TreeNode(preorder[0])           # first preorder value is always the root
    mid = inorder.index(preorder[0])       # find the root in inorder (splits left/right)
    # Everything left of the root in inorder is its left subtree, everything right its right.
    root.left = build_naive(preorder[1:mid+1], inorder[:mid])
    root.right = build_naive(preorder[mid+1:], inorder[mid+1:])
    return root

### Approach 2 — Hash Map + Pointer (optimal)

**Idea:** Precompute value → inorder index. Walk preorder with a single moving pointer; use index bounds instead of slicing.

**Time:** `O(n)`.

**Space:** `O(n)`.

In [ ]:
def build_fast(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    idx = {v: i for i, v in enumerate(inorder)}   # value -> its position in inorder (O(1))
    self_pre = [0]                         # our current position in preorder (in a list so it persists)
    def helper(lo, hi):                    # build the subtree covering inorder[lo..hi]
        if lo > hi:
            return None
        val = preorder[self_pre[0]]        # next preorder value is this subtree's root
        self_pre[0] += 1                   # advance the preorder pointer
        node = TreeNode(val)
        m = idx[val]                       # where the root splits inorder (instant lookup)
        node.left = helper(lo, m - 1)      # build the left part first (preorder does left first)
        node.right = helper(m + 1, hi)     # then the right part
        return node
    return helper(0, len(inorder) - 1)

In [ ]:
# Correctness check (rebuild, then re-extract the traversals)
tests = [([3,9,20,15,7],[9,3,15,20,7]), ([1,2],[2,1]), ([],[])]
for pre, ino in tests:
    t1, t2 = build_naive(pre, ino), build_fast(pre, ino)
    print(f"pre={pre}, ino={ino} -> preorder(rebuilt)={preorder(t1)}")
    assert preorder(t1) == pre and inorder(t1) == ino, "naive wrong"
    assert same_shape(t1, t2), "approaches disagree"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    t = build_balanced(n)
    return (preorder(t), inorder(t))
solutions = {
    "naive O(n^2)": build_naive,
    "fast  O(n)  ": build_fast,
}
sizes = [200, 400, 800, 1600]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Two orders pin down the tree:** preorder gives the root; inorder splits left/right.
- **Hash map to kill a repeated scan:** replacing `list.index` with a precomputed dict turns `O(n²)` into `O(n)`.
- **Signal:** "rebuild a tree from traversals".
- **Related problems:** Construct from Inorder+Postorder, Serialize/Deserialize.
- **Common pitfalls:** (1) slicing (costly + index confusion) — pass bounds instead; (2) advancing the preorder pointer incorrectly.